In [1]:
import os
from pathlib import Path

project_root = Path(r"c:\Users\pooja\OneDrive\Desktop\Ai healthcare").resolve()
print("Project root:", project_root)

for folder in [project_root / "datasets", project_root / "models", project_root / "scalers"]:
    folder.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import urllib.request

print("Libraries imported successfully")

Project root: C:\Users\pooja\OneDrive\Desktop\Ai healthcare
Libraries imported successfully


In [ ]:
# Download datasets only when missing, then inspect raw dataset files
candidate_urls = {
    'diabetes': [
        'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv',
        'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
    ],
    'heart': [
        'https://raw.githubusercontent.com/plotly/datasets/master/heart.csv',
        'https://raw.githubusercontent.com/selva86/datasets/master/Heart.csv',
        'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
    ],
    'kidney': [
        'https://raw.githubusercontent.com/avinashthu/CKD-Dataset/master/kidney_disease.csv',
        'https://raw.githubusercontent.com/ishandutta2007/Chronic-Kidney-Disease-Dataset/main/kidney_disease.csv'
    ]
}

for name, urls in candidate_urls.items():
    path = project_root / 'datasets' / f'{name}.csv'
    if path.exists():
        print(f'{name} dataset already exists: {path}')
        continue

    for url in urls:
        try:
            print(f'Trying {name} URL: {url}')
            urllib.request.urlretrieve(url, path)
            print(f'Successfully downloaded {name} dataset to {path}')
            break
        except Exception as ex:
            print(f'Failed {name} from {url}: {ex}')
    else:
        print(f'Could not download {name} dataset from candidate URLs')

print('\nDataset inspection:')
for name in ['diabetes', 'heart', 'kidney']:
    path = project_root / 'datasets' / f'{name}.csv'
    print('\n---', name, '---')
    if not path.exists():
        print('Missing dataset file:', path)
        continue
    try:
        sample = pd.read_csv(path, nrows=5)
        print('Shape:', sample.shape)
        print('Columns:', sample.columns.tolist())
        print(sample.head(5))
    except Exception as ex:
        print('Error reading', name, path, ex)


diabetes dataset already exists at C:\Users\pooja\OneDrive\Desktop\Ai healthcare\datasets\diabetes.csv
heart dataset already exists at C:\Users\pooja\OneDrive\Desktop\Ai healthcare\datasets\heart.csv
Trying kidney URL: https://raw.githubusercontent.com/avinashthu/CKD-Dataset/master/kidney_disease.csv
Failed kidney from https://raw.githubusercontent.com/avinashthu/CKD-Dataset/master/kidney_disease.csv: HTTP Error 404: Not Found
Trying kidney URL: https://raw.githubusercontent.com/ishandutta2007/Chronic-Kidney-Disease-Dataset/main/kidney_disease.csv
Failed kidney from https://raw.githubusercontent.com/ishandutta2007/Chronic-Kidney-Disease-Dataset/main/kidney_disease.csv: HTTP Error 404: Not Found
Could not download kidney dataset from candidate URLs
--- diabetes ---
Shape: (768, 9)
Columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
[{'Pregnancies': 6, 'Glucose': 148, 'BloodPressure': 72, 'SkinThickness': 3

In [20]:

# Check OpenML for heart disease and kidney disease datasets
from sklearn.datasets import fetch_openml

openml_names = ['heart-statlog', 'heart', 'diabetes', 'chronic_kidney_disease']
for name in openml_names:
    try:
        ds = fetch_openml(name=name, as_frame=True)
        print('OpenML', name, 'loaded:', ds.data.shape, 'target:', ds.target.name)
        print('Columns:', ds.data.columns.tolist()[:10], '...')
    except Exception as e:
        print('OpenML failed for', name, e)


OpenML heart-statlog loaded: (270, 13) target: class
Columns: ['age', 'sex', 'chest', 'resting_blood_pressure', 'serum_cholestoral', 'fasting_blood_sugar', 'resting_electrocardiographic_results', 'maximum_heart_rate_achieved', 'exercise_induced_angina', 'oldpeak'] ...
OpenML failed for heart Sparse ARFF datasets cannot be loaded with as_frame=True. Use as_frame=False or as_frame='auto' instead.
OpenML diabetes loaded: (768, 8) target: class
Columns: ['preg', 'plas', 'pres', 'skin', 'insu', 'mass', 'pedi', 'age'] ...


c:\Users\pooja\OneDrive\Desktop\Ai healthcare\.venv\Lib\site-packages\sklearn\datasets\_openml.py:328: UserWarning: Multiple active versions of the dataset matching the name diabetes exist. Versions may be fundamentally different, returning version 1. Available versions:
- version 1, status: active
  url: https://www.openml.org/search?type=data&id=37
- version 5, status: active
  url: https://www.openml.org/search?type=data&id=42608

  warn(warning_msg)


OpenML failed for chronic_kidney_disease No active dataset chronic_kidney_disease found.


In [ ]:
# Normalize and clean datasets, then write back fixed CSV files

def safe_read_dataset(path, expected_columns, target_column, alias_columns=None):
    """Read a dataset with or without headers and clean it."""
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path}")

    alias_columns = alias_columns or {}

    # Read raw strings to detect whether the first row is a header row
    raw = pd.read_csv(path, header=None, dtype=str, keep_default_na=False)
    first_row = [str(v).strip() for v in raw.iloc[0].tolist()]
    first_row_matches_header = any(
        item in expected_columns or item in alias_columns for item in first_row
    )

    if first_row_matches_header:
        df = pd.read_csv(path)
        df.columns = [str(c).strip() for c in df.columns]

        if df.iloc[0].astype(str).str.strip().tolist() == [str(col) for col in df.columns]:
            df = df.drop(index=0).reset_index(drop=True)
            print(f"Removed accidental repeated header row from {path.name}")
    else:
        df = pd.read_csv(path, header=None)
        df.columns = expected_columns
        print(f"No header row detected for {path.name}: assigned headers")

    rename_map = {col: alias_columns[col] for col in df.columns if col in alias_columns}
    if rename_map:
        df = df.rename(columns=rename_map)
        print(f"Renamed columns in {path.name}: {rename_map}")

    # Drop fully empty rows
    df = df.dropna(how='all')

    # Remove accidental repeated header rows anywhere in the dataset
    repeated_header = df[df.apply(
        lambda row: row.astype(str).str.strip().tolist() == [str(col) for col in df.columns],
        axis=1
    )]
    if len(repeated_header) > 0:
        print(f"Removing {len(repeated_header)} accidental repeated header rows from {path.name}")
        df = df.drop(repeated_header.index).reset_index(drop=True)

    # Remove duplicate rows
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        print(f"Removing {duplicate_count} duplicate rows from {path.name}")
        df = df.drop_duplicates().reset_index(drop=True)

    # Convert numeric columns safely
    for column in expected_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors='coerce')

    before_drop = len(df)
    df = df.dropna().reset_index(drop=True)
    dropped = before_drop - len(df)
    if dropped > 0:
        print(f"Dropped {dropped} invalid or missing rows from {path.name}")

    print(f"Read {path.name} | shape: {df.shape}")
    print('Columns:', df.columns.tolist())
    print(df.dtypes)
    print(df.head(5))
    print('Missing values:')
    print(df.isna().sum())
    print('Duplicate rows removed:', duplicate_count)

    missing = [c for c in expected_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns in {path.name}: {missing}")

    return df

# Define expected columns and targets for each dataset
expected_diabetes_columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Diabetes']
expected_heart_columns = ['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'Oldpeak', 'Slope', 'HeartDisease']
expected_kidney_columns = ['Age', 'BloodPressure', 'SpecificGravity', 'Albumin', 'Sugar', 'Creatinine', 'BloodUrea', 'Hemoglobin', 'WhiteBloodCells', 'RedBloodCells', 'Hypertension', 'Diabetes', 'KidneyDisease']

# Read or generate diabetes dataset
try:
    diabetes_path = project_root / 'datasets' / 'diabetes.csv'
    diabetes_df = safe_read_dataset(
        diabetes_path,
        expected_diabetes_columns,
        'Diabetes',
        alias_columns={'Outcome': 'Diabetes', 'outcome': 'Diabetes'}
    )
    diabetes_df.to_csv(diabetes_path, index=False)
except Exception as e:
    print(f"Error loading diabetes dataset: {e}")
    raise

# Read or generate heart dataset
try:
    heart_path = project_root / 'datasets' / 'heart.csv'
    heart_df = safe_read_dataset(
        heart_path,
        expected_heart_columns,
        'HeartDisease',
        alias_columns={'Target': 'HeartDisease', 'target': 'HeartDisease'}
    )
    heart_df.to_csv(heart_path, index=False)
except Exception as e:
    print(f"Error loading heart dataset: {e}")
    raise

# Read or generate kidney dataset
try:
    kidney_path = project_root / 'datasets' / 'kidney.csv'
    if not kidney_path.exists():
        print('Generating synthetic kidney dataset because kidney.csv is missing')
        np.random.seed(42)
        n_samples = 1200
        kidney_df = pd.DataFrame({
            'Age': np.random.randint(20, 85, n_samples),
            'BloodPressure': np.random.uniform(70, 180, n_samples),
            'SpecificGravity': np.random.choice([1.005, 1.010, 1.015, 1.020, 1.025, 1.030], n_samples),
            'Albumin': np.random.choice([0, 1, 2, 3, 4, 5], n_samples),
            'Sugar': np.random.choice([0, 1, 2, 3, 4, 5], n_samples),
            'Creatinine': np.random.uniform(0.5, 8.0, n_samples),
            'BloodUrea': np.random.uniform(20, 150, n_samples),
            'Hemoglobin': np.random.uniform(7, 17, n_samples),
            'WhiteBloodCells': np.random.uniform(3000, 20000, n_samples),
            'RedBloodCells': np.random.uniform(2, 6, n_samples),
            'Hypertension': np.random.choice([0, 1], n_samples),
            'Diabetes': np.random.choice([0, 1], n_samples)
        })
        kidney_df['KidneyDisease'] = (
            (kidney_df['Creatinine'] > 1.5).astype(int) * 0.35 +
            (kidney_df['BloodUrea'] > 100).astype(int) * 0.25 +
            (kidney_df['Hemoglobin'] < 10).astype(int) * 0.2 +
            (kidney_df['Hypertension'] == 1).astype(int) * 0.1 +
            (kidney_df['Diabetes'] == 1).astype(int) * 0.1 +
            (np.random.rand(n_samples) * 0.05)
        ) > 0.5
        kidney_df['KidneyDisease'] = kidney_df['KidneyDisease'].astype(int)
        kidney_df.to_csv(kidney_path, index=False)
        print('Generated synthetic kidney dataset at', kidney_path)

    kidney_df = safe_read_dataset(
        kidney_path,
        expected_kidney_columns,
        'KidneyDisease',
        alias_columns={'class': 'KidneyDisease', 'Class': 'KidneyDisease'}
    )
    kidney_df.to_csv(kidney_path, index=False)
except Exception as e:
    print(f"Error loading kidney dataset: {e}")
    raise


ValueError: invalid literal for int() with base 10: 'Target'

In [ ]:
# Explore and clean the heart disease dataset

heart_df.head(), heart_df.info(), heart_df.isna().sum()

In [ ]:
# Explore and clean the kidney disease dataset

kidney_df.head(), kidney_df.info(), kidney_df.isna().sum()

In [ ]:
# Train models using cleaned datasets and verify saved files

models_dir = project_root / 'models'
scalers_dir = project_root / 'scalers'
models_dir.mkdir(parents=True, exist_ok=True)
scalers_dir.mkdir(parents=True, exist_ok=True)

# Training utility

def train_and_save_model(df, feature_columns, target_column, model_name):
    X = df[feature_columns]
    y = df[target_column].astype(int)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_scaled, y)

    model_path = models_dir / f'{model_name}_model.pkl'
    scaler_path = scalers_dir / f'{model_name}_scaler.pkl'
    joblib.dump(model, model_path)
    joblib.dump(scaler, scaler_path)

    y_pred = model.predict(X_scaled)
    print(f'--- {model_name} ---')
    print('Shape:', X.shape)
    print('Accuracy:', accuracy_score(y, y_pred))
    print('Precision:', precision_score(y, y_pred, average='weighted'))
    print('Recall:', recall_score(y, y_pred, average='weighted'))
    print('F1 Score:', f1_score(y, y_pred, average='weighted'))
    print(f'✓ {model_name.replace("_", " ").title()} model saved')

    return model_path, scaler_path

# Diabetes model
feature_columns = ['Age', 'BMI', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'DiabetesPedigreeFunction', 'Pregnancies']
diabetes_model_path, diabetes_scaler_path = train_and_save_model(diabetes_df, feature_columns, 'Diabetes', 'diabetes')

# Heart disease model
feature_columns = ['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'Oldpeak', 'Slope']
heart_model_path, heart_scaler_path = train_and_save_model(heart_df, feature_columns, 'HeartDisease', 'heart_disease')

# Kidney disease model
feature_columns = ['Age', 'BloodPressure', 'SpecificGravity', 'Albumin', 'Sugar', 'Creatinine', 'BloodUrea', 'Hemoglobin', 'WhiteBloodCells', 'RedBloodCells', 'Hypertension', 'Diabetes']
kidney_model_path, kidney_scaler_path = train_and_save_model(kidney_df, feature_columns, 'KidneyDisease', 'kidney_disease')

# Verify each saved model by loading them back
try:
    joblib.load(diabetes_model_path)
    joblib.load(diabetes_scaler_path)
    print('✓ Diabetes model saved')
except Exception as e:
    print('Failed to verify diabetes model:', e)

try:
    joblib.load(heart_model_path)
    joblib.load(heart_scaler_path)
    print('✓ Heart Disease model saved')
except Exception as e:
    print('Failed to verify heart disease model:', e)

try:
    joblib.load(kidney_model_path)
    joblib.load(kidney_scaler_path)
    print('✓ Kidney Disease model saved')
except Exception as e:
    print('Failed to verify kidney disease model:', e)
